# Phi3.5 4B Manual Soft Prompt Tuning

Manual soft prompt implementation for Phi-3.5-Vision on RSICD dataset.

by [Gayanuka Amarasuriya](https://gayanukaa.github.io/)


In [ ]:
!pip install -q torch transformers datasets accelerate scikit-learn pycocoevalcap

In [ ]:
import torch
import torch.nn as nn
import json
import time
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice

## Configuration


In [ ]:
MODEL_NAME = "microsoft/Phi-3.5-vision-instruct"
DATASET_NAME = "arampacha/rsicd"
MAX_LENGTH = 128
PROMPT_LENGTH = 4
BATCH_SIZE = 2
NUM_EPOCHS = 3
OUTPUT_DIR = "./vlm_prompt_output"
device = "cuda" if torch.cuda.is_available() else "cpu"

## Manual Soft Prompt Implementation


In [ ]:
class VLMSoftPromptTuning(nn.Module):
    def __init__(self, base_model, num_tokens=4):
        super().__init__()
        self.base_model = base_model
        self.num_tokens = num_tokens

        # Get embedding dimension from model
        self.embed_dim = base_model.get_input_embeddings().weight.shape[1]

        # Initialize learnable soft prompts
        self.soft_prompts = nn.Parameter(
            torch.randn(num_tokens, self.embed_dim) * 0.1
        )

        # Freeze base model parameters
        for param in self.base_model.parameters():
            param.requires_grad = False

    def forward(self, input_ids, pixel_values=None, attention_mask=None, labels=None, **kwargs):
        batch_size = input_ids.shape[0]

        # Get input embeddings
        input_embeds = self.base_model.get_input_embeddings()(input_ids)

        # Expand soft prompts for batch
        soft_prompts_batch = self.soft_prompts.unsqueeze(0).expand(batch_size, -1, -1)

        # Concatenate soft prompts with input embeddings
        combined_embeds = torch.cat([soft_prompts_batch, input_embeds], dim=1)

        # Extend attention mask for soft prompts
        if attention_mask is not None:
            prompt_mask = torch.ones(batch_size, self.num_tokens, device=attention_mask.device)
            attention_mask = torch.cat([prompt_mask, attention_mask], dim=1)

        # Adjust labels if provided
        if labels is not None:
            # Pad labels with -100 for soft prompt positions
            label_pad = torch.full((batch_size, self.num_tokens), -100, device=labels.device)
            labels = torch.cat([label_pad, labels], dim=1)

        return self.base_model(
            inputs_embeds=combined_embeds,
            pixel_values=pixel_values,
            attention_mask=attention_mask,
            labels=labels,
            **kwargs
        )

    def generate(self, input_ids, pixel_values=None, attention_mask=None, **kwargs):
        batch_size = input_ids.shape[0]

        # Get input embeddings
        input_embeds = self.base_model.get_input_embeddings()(input_ids)

        # Expand soft prompts for batch
        soft_prompts_batch = self.soft_prompts.unsqueeze(0).expand(batch_size, -1, -1)

        # Concatenate soft prompts with input embeddings
        combined_embeds = torch.cat([soft_prompts_batch, input_embeds], dim=1)

        # Extend attention mask for soft prompts
        if attention_mask is not None:
            prompt_mask = torch.ones(batch_size, self.num_tokens, device=attention_mask.device)
            attention_mask = torch.cat([prompt_mask, attention_mask], dim=1)

        return self.base_model.generate(
            inputs_embeds=combined_embeds,
            pixel_values=pixel_values,
            attention_mask=attention_mask,
            **kwargs
        )

    def print_trainable_parameters(self):
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.parameters())
        print(f"Trainable parameters: {trainable_params:,}")
        print(f"Total parameters: {total_params:,}")
        print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

## Load Model and Tokenizer


In [ ]:
print(f"Loading {MODEL_NAME}...")

processor = AutoProcessor.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

# Move to device
base_model = base_model.to(device)

# Create soft prompt model
model = VLMSoftPromptTuning(base_model, PROMPT_LENGTH)
model.print_trainable_parameters()

## Load Dataset


In [ ]:
dataset = load_dataset(DATASET_NAME)

# Split as requested
train_data = dataset["train"].select(range(1000))
eval_data = dataset["valid"].select(range(200))
test_data = dataset["test"].select(range(10))

print(f"Training samples: {len(train_data)}")
print(f"Evaluation samples: {len(eval_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Sample: {train_data[0]}")

## Data Preprocessing


In [ ]:
def preprocess_function(examples):
    # Prepare images and text
    images = examples["image"]
    captions = [caption[0] for caption in examples["captions"]]  # Take first caption

    # Create conversation format for VLM
    conversations = []
    for caption in captions:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": "Describe this satellite image."}
                ]
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": caption}]
            }
        ]
        conversations.append(messages)

    # Process with processor
    processed_inputs = []
    for i, conv in enumerate(conversations):
        # Apply chat template
        text = processor.apply_chat_template(conv, tokenize=False)

        # Process text and image
        inputs = processor(
            text=text,
            images=images[i],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH
        )

        processed_inputs.append({
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "pixel_values": inputs["pixel_values"].squeeze(),
            "labels": inputs["input_ids"].squeeze()
        })

    # Combine all processed inputs
    batch = {
        "input_ids": torch.stack([item["input_ids"] for item in processed_inputs]),
        "attention_mask": torch.stack([item["attention_mask"] for item in processed_inputs]),
        "pixel_values": torch.stack([item["pixel_values"] for item in processed_inputs]),
        "labels": torch.stack([item["labels"] for item in processed_inputs])
    }

    return batch

# Preprocess datasets
train_dataset = train_data.map(
    preprocess_function,
    batched=True,
    remove_columns=train_data.column_names,
    batch_size=8  # Process in smaller batches for memory
)

eval_dataset = eval_data.map(
    preprocess_function,
    batched=True,
    remove_columns=eval_data.column_names,
    batch_size=8
)

test_dataset = test_data.map(
    preprocess_function,
    batched=True,
    remove_columns=test_data.column_names,
    batch_size=8
)

# Set format
train_dataset.set_format("torch")
eval_dataset.set_format("torch")
test_dataset.set_format("torch")

print(f"Processed training samples: {len(train_dataset)}")
print(f"Sample keys: {list(train_dataset[0].keys())}")
print(f"Input shape: {train_dataset[0]['input_ids'].shape}")
print(f"Pixel values shape: {train_dataset[0]['pixel_values'].shape}")

## Evaluation Metrics


In [ ]:
# Initialize scorers
cider_scorer = Cider()
spice_scorer = Spice()

def compute_cosine_similarity(predictions, references):
    vectorizer = TfidfVectorizer()
    all_texts = predictions + [ref[0] for ref in references]

    try:
        tfidf_matrix = vectorizer.fit_transform(all_texts)
        pred_vectors = tfidf_matrix[:len(predictions)]
        ref_vectors = tfidf_matrix[len(predictions):]

        similarities = []
        for i in range(len(predictions)):
            sim = cosine_similarity(pred_vectors[i], ref_vectors[i])[0][0]
            similarities.append(sim)

        return np.mean(similarities)
    except:
        return 0.0

def compute_all_metrics(predictions, references):
    results = {}

    # Format for pycocoevalcap
    gts = {i: ref_list for i, ref_list in enumerate(references)}
    res = {i: [pred] for i, pred in enumerate(predictions)}

    # CIDEr
    try:
        cider_score, _ = cider_scorer.compute_score(gts, res)
        results['CIDEr'] = cider_score
    except Exception as e:
        print(f"CIDEr failed: {e}")
        results['CIDEr'] = 0.0

    # SPICE
    try:
        spice_score, _ = spice_scorer.compute_score(gts, res)
        results['SPICE'] = spice_score
    except Exception as e:
        print(f"SPICE failed: {e}")
        results['SPICE'] = 0.0

    # Cosine similarity
    results['Cosine_Similarity'] = compute_cosine_similarity(predictions, references)

    return results

## Training Setup


In [ ]:
class SoftPromptTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        outputs = model(**inputs)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=1e-3,
    warmup_steps=50,
    logging_steps=10,
    eval_steps=100,
    save_steps=200,
    evaluation_strategy="steps",
    save_total_limit=1,
    fp16=True,
    report_to="none",
    remove_unused_columns=True,
    dataloader_drop_last=True
)

# Create trainer
trainer = SoftPromptTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer
)

## Train Model


In [ ]:
print("Starting training...")
trainer.train()
print("Training completed!")

## Test Generation


In [ ]:
def generate_caption(model, tokenizer, prompt, max_length=64):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Remove the input prompt from generated text
    generated = generated.replace(prompt, "").strip()
    return generated

# Test on a few samples
print("Testing generation...")
for i in range(3):
    prompt = "Describe this satellite image: "
    reference = test_data[i]["captions"][0]

    generated = generate_caption(model, tokenizer, prompt)

    print(f"\nSample {i+1}:")
    print(f"Generated: {generated}")
    print(f"Reference: {reference}")
    print("-" * 50)

## Full Evaluation


In [ ]:
print("Running full evaluation on test set...")

predictions = []
references = []
inference_times = []
vram_usage = []

model.eval()

for i in range(len(test_data)):
    sample = test_data[i]
    prompt = "Describe this satellite image: "
    reference = sample["captions"][0]

    # Measure inference time
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    start_time = time.time()
    generated = generate_caption(model, tokenizer, prompt)
    end_time = time.time()

    inference_time = end_time - start_time

    # Measure VRAM
    if torch.cuda.is_available():
        vram = torch.cuda.max_memory_allocated() / 1e9  # GB
    else:
        vram = 0

    predictions.append(generated)
    references.append([reference])
    inference_times.append(inference_time)
    vram_usage.append(vram)

    print(f"Sample {i+1}/{len(test_data)} - Time: {inference_time:.3f}s, VRAM: {vram:.2f}GB")

# Compute metrics
print("\nComputing metrics...")
metrics = compute_all_metrics(predictions, references)

# Performance stats
avg_inference_time = np.mean(inference_times)
avg_vram = np.mean(vram_usage)

print("\n=== Results ===")
print(f"CIDEr: {metrics['CIDEr']:.4f}")
print(f"SPICE: {metrics['SPICE']:.4f}")
print(f"Cosine Similarity: {metrics['Cosine_Similarity']:.4f}")
print(f"Avg Inference Time: {avg_inference_time:.3f}s")
print(f"Avg VRAM Usage: {avg_vram:.2f}GB")

## Save Results


In [ ]:
# Save soft prompts
torch.save({
    'soft_prompts': model.soft_prompts,
    'model_name': MODEL_NAME,
    'num_tokens': PROMPT_LENGTH
}, f"{OUTPUT_DIR}/soft_prompts.pt")

# Save evaluation results
results = {
    'model': MODEL_NAME,
    'dataset': DATASET_NAME,
    'metrics': {
        'cider': metrics['CIDEr'],
        'spice': metrics['SPICE'],
        'cosine_similarity': metrics['Cosine_Similarity'],
        'avg_inference_time': avg_inference_time,
        'avg_vram_usage': avg_vram
    },
    'predictions': predictions,
    'references': [ref[0] for ref in references]
}

with open(f"{OUTPUT_DIR}/evaluation_results.json", 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {OUTPUT_DIR}/")
print(f"Soft prompts saved to {OUTPUT_DIR}/soft_prompts.pt")

## Load Saved Model


In [ ]:
def load_soft_prompt_model(checkpoint_path, model_name):
    # Load base model
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    ).to(device)

    # Load checkpoint
    checkpoint = torch.load(checkpoint_path)

    # Create soft prompt model
    model = VLMSoftPromptTuning(base_model, checkpoint['num_tokens'])

    # Load soft prompts
    model.soft_prompts.data = checkpoint['soft_prompts']

    return model

# Example usage (uncomment to test):
# loaded_model = load_soft_prompt_model(f"{OUTPUT_DIR}/soft_prompts.pt", MODEL_NAME)
# print("Model loaded successfully!")